# LaTeX export — figures and tables for the thesis document

Exports a figure and a table straight into the sibling thesis repository, so a chart in the written thesis is a regenerated artifact rather than a screenshot.

**Inputs:** joined train/test feature artifacts and their metadata contract, plus the corresponding all-station observation artifacts (Stages 2–3 must have run)
**Outputs:** `../uas-master-thesis/figures/*.pdf` and `../uas-master-thesis/tables/*.tex` — **written outside this repository**

This notebook is not part of the `01` → `06` chain and has no `make` target; run it by hand while writing. The thesis preamble needs `\usepackage{booktabs}` for the exported table to compile.

In [ ]:
%load_ext autoreload
%autoreload 2

## Setup

Pins the same Stage-3 artifact paths and cohort constants every stage-4 notebook uses. The completeness figure reads the joined feature artifacts, so it includes only stations retained by the target-range overlap filter. `THESIS_TEXTWIDTH_IN` is the thesis body text width (456.25555 pt): authoring the figure at that width means `width=\textwidth` scales it by 1.0 and the in-figure fonts land at their intended size.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import pandas as pd
from IPython.display import display

from src.config import (
    EMBARGO_HOURS,
    FORECAST_HORIZON_HOURS,
    INITIAL_TRAIN_FRACTION,
    MATPLOTLIB_STYLE,
    N_VALIDATION_FOLDS,
    TARGET_STATION_ID,
    WATER_LEVEL_ALARM_THRESHOLD_CM,
    THESIS_TEXTWIDTH_IN,
    WEATHER_VARIABLES,
)
plt.style.use(MATPLOTLIB_STYLE)
from src.dataset import load_joined_dataset
from src.latex_export import save_figure, save_table

PROCESSED_DIR = Path("data/processed/joined")
METADATA_PATH = PROCESSED_DIR / "all_stations_feature_metadata.json"
train_path = PROCESSED_DIR / "all_stations_train_features.parquet"
test_path = PROCESSED_DIR / "all_stations_test_features.parquet"
raw_train_path = PROCESSED_DIR / "all_stations_train.parquet"
raw_test_path = PROCESSED_DIR / "all_stations_test.parquet"
WATER_LEVEL_COLUMN = f"{TARGET_STATION_ID}__water_level"

## Load the joined dataset

`load_joined_dataset()` is the repo's only sanctioned entry point into the Stage-3 artifacts: it validates the horizon, target station, and column contracts before returning anything, so a drifted contract fails here instead of silently producing a figure of the wrong thing.

In [ ]:
dataset = load_joined_dataset(
    METADATA_PATH,
    train_path,
    test_path,
    station_id=TARGET_STATION_ID,
    forecast_horizon_hours=FORECAST_HORIZON_HOURS,
    weather_variables=WEATHER_VARIABLES,
    initial_train_fraction=INITIAL_TRAIN_FRACTION,
    n_validation_folds=N_VALIDATION_FOLDS,
    embargo_rows=EMBARGO_HOURS,
)
water_level = dataset.target_context_series[WATER_LEVEL_COLUMN]
water_level_columns = [
    column
    for column in dataset.contract.predictor_columns
    if column.endswith("__water_level")
]

## Figure — full-history water level and alarm threshold

The full raw train and sealed-test observation artifacts are combined before aggregation, so this figure covers the entire available period rather than only the eligible modeling cohort. The daily mean water level and weekly precipitation total share one time axis; separate y-axes preserve their different units.

In [ ]:
PRECIPITATION_COLUMN = f"{TARGET_STATION_ID}__precipitation"
full_history = pd.concat(
    [
        pd.read_parquet(
            raw_train_path,
            columns=["timestamp", WATER_LEVEL_COLUMN, PRECIPITATION_COLUMN],
        ),
        pd.read_parquet(
            raw_test_path,
            columns=["timestamp", WATER_LEVEL_COLUMN, PRECIPITATION_COLUMN],
        ),
    ],
    ignore_index=True,
).assign(timestamp=lambda frame: pd.to_datetime(frame["timestamp"], utc=True))
full_history = full_history.sort_values("timestamp").set_index("timestamp")
split_timestamp = pd.to_datetime(
    pd.read_parquet(raw_test_path, columns=["timestamp"])["timestamp"],
    utc=True,
).min()
daily_water_level = full_history[WATER_LEVEL_COLUMN].resample("D").max()
weekly_precipitation = full_history[PRECIPITATION_COLUMN].resample("W").sum()

fig, water_axis = plt.subplots(
    figsize=(THESIS_TEXTWIDTH_IN, THESIS_TEXTWIDTH_IN * 0.55)
)
precipitation_axis = water_axis.twinx()
daily_line, = water_axis.plot(
    daily_water_level.index, daily_water_level, label="Daily max"
)
threshold_line = water_axis.axhline(
    WATER_LEVEL_ALARM_THRESHOLD_CM,
    color="C2",
    linestyle="--",
    label=f"Alarm threshold ({WATER_LEVEL_ALARM_THRESHOLD_CM:.0f} cm)",
)
weekly_bars = precipitation_axis.bar(
    weekly_precipitation.index,
    weekly_precipitation,
    width=10,
    color="C1",
    alpha=0.8,
    label="Weekly total",
)
split_line = water_axis.axvline(
    split_timestamp,
    color="black",
    linestyle=":",
    label="Train/test split",
)
water_axis.set_xlabel("Date")
water_axis.set_ylabel("Water level (cm)")
precipitation_axis.set_ylabel("Weekly precip. (mm)")
water_axis.set_title(f"Korneuburg water level ({TARGET_STATION_ID})")
water_axis.grid(alpha=0.25)
fig.tight_layout(rect=(0, 0.18, 1, 1))
fig.legend(
    handles=[daily_line, threshold_line, weekly_bars, split_line],
    loc="lower center",
    bbox_to_anchor=(0.5, 0.02),
    ncol=2
)
save_figure(
    fig,
    "target_water_level_history",
    caption="Daily max Korneuburg water level at station 207241-at over the full available period with weekly precipitation totals and the train/test split.",
)

## Figure — monthly data completeness by station

The heatmap shows the monthly percentage of available water level measurements for each station retained in the joined dataset. Interpolated measurements are counted as missing, so only directly observed measurements contribute to the displayed share. The bottom row shows the percentage of rows eligible for modeling; eligibility follows the joined feature contract.

In [ ]:
joined_columns = list(
    dict.fromkeys(
        [
            "timestamp",
            *water_level_columns,
            *(
                column.removesuffix("__water_level") + "__imputed"
                for column in water_level_columns
            ),
            dataset.contract.target_valid_column,
            *dataset.contract.predictor_columns,
            *dataset.contract.target_columns,
        ]
    )
)
joined_features = pd.concat(
    [
        pd.read_parquet(train_path, columns=joined_columns),
        pd.read_parquet(test_path, columns=joined_columns),
    ],
    ignore_index=True,
)
joined_features["timestamp"] = pd.to_datetime(
    joined_features["timestamp"], utc=True
)
joined_features["month"] = (
    joined_features["timestamp"].dt.tz_localize(None).dt.to_period("M")
)
station_ids = [
    column.removesuffix("__water_level") for column in water_level_columns
]
non_missing_percentages = {}
total_non_missing_percentages = {}
for station_id in station_ids:
    water_level_column = f"{station_id}__water_level"
    imputed_column = f"{station_id}__imputed"
    missing = (
        joined_features[water_level_column].isna()
        | joined_features[imputed_column].eq(True)
    )
    non_missing = ~missing
    non_missing_percentages[station_id] = (
        pd.DataFrame({"month": joined_features["month"], "non_missing": non_missing})
        .groupby("month")["non_missing"]
        .mean()
        .mul(100)
    )
    total_non_missing_percentages[station_id] = non_missing.mean() * 100

all_months = pd.period_range(
    joined_features["month"].min(),
    joined_features["month"].max(),
    freq="M",
)
missing_table = pd.DataFrame(
    {
        station_id: percentages.reindex(all_months)
        for station_id, percentages in non_missing_percentages.items()
    }
).T.reindex(station_ids)
eligible = (
    joined_features[dataset.contract.target_valid_column].eq(True)
    & joined_features[list(dataset.contract.predictor_columns)]
    .notna()
    .all(axis=1)
    & joined_features[list(dataset.contract.target_columns)].notna().all(axis=1)
)
total_eligible_percentage = eligible.mean() * 100
eligible_percentages = (
    pd.DataFrame({"month": joined_features["month"], "eligible": eligible})
    .groupby("month")["eligible"]
    .mean()
    .mul(100)
    .reindex(all_months)
)
eligible_row = pd.DataFrame(
    [eligible_percentages.to_numpy()],
    index=["Eligible rows"],
    columns=all_months,
)
completeness_table = pd.concat([missing_table, eligible_row])

cmap = plt.get_cmap("viridis").copy()
cmap.set_bad("white")
fig = plt.figure(figsize=(THESIS_TEXTWIDTH_IN, THESIS_TEXTWIDTH_IN * 0.58))
grid = fig.add_gridspec(
    2,
    3,
    height_ratios=(20, 1.2),
    width_ratios=(20, 4, 1),
    wspace=0.05,
    hspace=0.9,
)
axis = fig.add_subplot(grid[0, 0])
total_axis = fig.add_subplot(grid[0, 1], sharey=axis)
colorbar_axis = fig.add_subplot(grid[1, 0])
image = axis.imshow(
    completeness_table.to_numpy(dtype=float),
    aspect="auto",
    interpolation="none",
    cmap=cmap,
    vmin=0,
    vmax=100,
)
total_axis.set_xlim(0, 1)
total_axis.set_xticks([])
total_axis.tick_params(axis="y", left=False, labelleft=False)
total_axis.text(
    0.5,
    1.02,
    "Total",
    transform=total_axis.transAxes,
    ha="center",
    va="bottom",
    fontweight="bold",
)
for row, station_id in enumerate(station_ids):
    total_axis.text(
        0.5,
        row,
        f"{total_non_missing_percentages[station_id]:.1f}%",
        ha="center",
        va="center",
    )
total_axis.text(
    0.5,
    len(station_ids),
    f"{total_eligible_percentage:.1f}%",
    ha="center",
    va="center",
)
axis.set_title("Available water level\nmeasurements by month", loc="left")
axis.set_yticks(range(len(completeness_table)))
axis.set_yticklabels(completeness_table.index)
for label in axis.get_yticklabels():
    if label.get_text() == TARGET_STATION_ID:
        label.set_fontweight("bold")
axis.set_ylabel("Station / summary")
axis.axhline(len(station_ids) - 0.5, color="white", linewidth=1.5)
year_positions = [
    position
    for position, month in enumerate(all_months)
    if month.month == 1
]
axis.set_xticks(year_positions)
axis.set_xticklabels(
    [str(all_months[position].year) for position in year_positions],
    rotation=45,
    ha="right",
)
axis.set_xlabel("Month")
colorbar = fig.colorbar(
    image,
    cax=colorbar_axis,
    orientation="horizontal",
)
colorbar.set_label("Available water level measurements (%)", labelpad=2)
for spine in colorbar_axis.spines.values():
    spine.set_visible(True)
    spine.set_color("black")
    spine.set_linewidth(0.8)
# total_axis.axhline(len(station_ids) - 0.5, color="0.5", linewidth=1.0)
fig.subplots_adjust(left=0.27, right=0.98, top=0.86, bottom=0.25)
legend_bbox = colorbar_axis.get_position()
legend_frame = FancyBboxPatch(
    (legend_bbox.x0 - 0.025, legend_bbox.y0 - 0.12),
    legend_bbox.width + 0.05,
    legend_bbox.height + 0.12,
    boxstyle="round,pad=0.01,rounding_size=0.015",
    transform=fig.transFigure,
    facecolor="white",
    edgecolor=plt.rcParams["legend.edgecolor"],
    linewidth=0.8,
    alpha=plt.rcParams["legend.framealpha"],
    zorder=0,
)
fig.add_artist(legend_frame)
save_figure(
    fig,
    "station_monthly_data_completeness",
    caption="Monthly available water level measurements and eligible rows for upstream stations (interpolated measurements count as missing).",
)

## Figure — cross-correlation between upstream stations and Korneuburg

For each water level column present in the joined feature Parquet files, this heatmap computes the pairwise-complete Pearson correlation with Korneuburg after shifting the upstream series by 0–72 hours. A positive lag therefore compares an upstream measurement with Korneuburg's level that many hours later; the marked maximum is a rough indication of propagation delay, not causal evidence.

In [ ]:
LAG_SCAN_HOURS = 72
feature_water_level_columns = [
    column
    for column in joined_features.columns
    if column.endswith("__water_level")
]
assert WATER_LEVEL_COLUMN in feature_water_level_columns
upstream_station_ids = [
    column.removesuffix("__water_level")
    for column in feature_water_level_columns
    if column != WATER_LEVEL_COLUMN
]
if not upstream_station_ids:
    raise ValueError("The joined feature artifacts contain no upstream water level columns")

correlation_input = joined_features.sort_values("timestamp").reset_index(drop=True)
target_series = correlation_input[WATER_LEVEL_COLUMN]
lag_hours = range(LAG_SCAN_HOURS + 1)
cross_correlation = pd.DataFrame(index=upstream_station_ids, columns=lag_hours, dtype=float)
for station_id in upstream_station_ids:
    upstream_series = correlation_input[f"{station_id}__water_level"]
    cross_correlation.loc[station_id] = [
        upstream_series.shift(lag).corr(target_series) for lag in lag_hours
    ]

fig, ax = plt.subplots(
    figsize=(THESIS_TEXTWIDTH_IN, THESIS_TEXTWIDTH_IN * 0.48)
)
image = ax.imshow(
    cross_correlation.to_numpy(),
    aspect="auto",
    interpolation="none",
    # cmap="PuOr",
    vmin=0,
    vmax=1,
)
ax.set_yticks(range(len(upstream_station_ids)))
ax.set_yticklabels(upstream_station_ids)
lag_ticks = list(range(0, LAG_SCAN_HOURS + 1, 6))
ax.set_xticks(lag_ticks)
ax.set_xticklabels(lag_ticks)
ax.set_xlabel("Lag before Korneuburg (h)")
ax.set_ylabel("Upstream station")
ax.set_title("Upstream water level correlation with Korneuburg")
for row, station_id in enumerate(upstream_station_ids):
    best_lag = cross_correlation.loc[station_id].idxmax()
    if pd.notna(best_lag):
        print(f"Station {station_id} has best correlation at lag {best_lag} hours")
        ax.plot(
            best_lag,
            row,
            marker="o",
            markerfacecolor="none",
            markeredgecolor="black",
            markersize=5,
            markeredgewidth=0.8,
        )
colorbar = fig.colorbar(image, ax=ax, pad=0.02)
colorbar.set_label("Pearson correlation")
fig.tight_layout()
save_figure(
    fig,
    "upstream_cross_correlation_heatmap",
    caption=f"Pearson cross-correlation between upstream stations and Korneuburg for lags of 0-{LAG_SCAN_HOURS} hours. Positive lag means the upstream measurement leads Korneuburg; circles mark each station's peak.",
)

## Figure — weather aggregates and future water level change

This RQ2 figure reproduces the descriptive Spearman correlations from the single-river EDA for the requested 6-, 24-, and 72-hour trailing precipitation sums and temperature means. Each cell relates the weather aggregate at issue time to the target station's water level change at the indicated future horizon; rows are paired only where both quantities are available.

In [ ]:
RELATIONSHIP_WINDOWS = (6, 24, 72)
RELATIONSHIP_HORIZONS = (1, 6, 12, 24, 48)
relationship_input = joined_features.sort_values("timestamp").reset_index(drop=True)
target_level = relationship_input[WATER_LEVEL_COLUMN]
future_changes = {
    horizon: target_level.shift(-horizon) - target_level
    for horizon in RELATIONSHIP_HORIZONS
}
relationship_records = []
for weather_label, column_prefix in [
    ("Percip.", "precipitation_rolling_sum"),
    ("Temp.", "temperature_2m_rolling_mean"),
]:
    for window in RELATIONSHIP_WINDOWS:
        aggregate_column = (
            f"{TARGET_STATION_ID}__{column_prefix}_{window}h"
        )
        row_label = f"{weather_label} last {window} h"
        for horizon, future_change in future_changes.items():
            pair = pd.concat(
                [
                    relationship_input[aggregate_column],
                    future_change.rename("future_change"),
                ],
                axis=1,
            ).dropna()
            relationship_records.append(
                {
                    "weather_aggregate": row_label,
                    "horizon_hours": horizon,
                    "spearman_rho": pair[aggregate_column].corr(
                        pair["future_change"], method="spearman"
                    ),
                }
            )
weather_relationships = pd.DataFrame(relationship_records)
relationship_matrix = weather_relationships.pivot(
    index="weather_aggregate",
    columns="horizon_hours",
    values="spearman_rho",
).reindex(
    index=[
        f"{weather} last {window} h"
        for weather in ("Percip.", "Temp.")
        for window in RELATIONSHIP_WINDOWS
    ],
    columns=RELATIONSHIP_HORIZONS,
)

fig, ax = plt.subplots(
    figsize=(THESIS_TEXTWIDTH_IN, THESIS_TEXTWIDTH_IN * 0.48)
)
image = ax.imshow(
    relationship_matrix.to_numpy(),
    aspect="auto",
    interpolation="none",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
)
ax.set_xticks(range(len(RELATIONSHIP_HORIZONS)))
ax.set_xticklabels([f"+{horizon} h" for horizon in RELATIONSHIP_HORIZONS])
ax.xaxis.tick_top()
ax.xaxis.set_label_position("top")
ax.set_xlabel("Future water level change", labelpad=8)
ax.set_yticks(range(len(relationship_matrix)))
ax.set_yticklabels(relationship_matrix.index)
ax.set_ylabel("Weather aggregate")
ax.set_title("Weather aggregates and future water level change", pad=34)
ax.axhline(2.5, color="white", linewidth=1.5)
for row in range(relationship_matrix.shape[0]):
    for column in range(relationship_matrix.shape[1]):
        value = relationship_matrix.iloc[row, column]
        ax.text(
            column,
            row,
            f"{value:.2f}",
            ha="center",
            va="center",
        )
colorbar = fig.colorbar(image, ax=ax, pad=0.02)
colorbar.set_label("Spearman correlation")
fig.tight_layout()
save_figure(
    fig,
    "weather_future_change_spearman",
    caption="Spearman correlations between precipitation sums or temperature means and future water level changes at the target station.",
)

## Figure — observed water level distribution

A violin plot with an overlaid boxplot of every observed water level at the target station, authored at exactly the thesis text width. `save_figure` writes the PDF and prints the `figure` float; the figure stays open so it also renders inline here.

In [ ]:
observed_water_level = water_level.dropna()
quartiles = observed_water_level.quantile([0.25, 0.50, 0.75])
above_alarm_percentage = (
    observed_water_level.gt(WATER_LEVEL_ALARM_THRESHOLD_CM).mean() * 100
)
above_alarm_count = observed_water_level.gt(WATER_LEVEL_ALARM_THRESHOLD_CM).sum()
fig, ax = plt.subplots(figsize=(THESIS_TEXTWIDTH_IN, THESIS_TEXTWIDTH_IN * 0.55))
violin = ax.violinplot(
    observed_water_level,
    positions=[1],
    orientation="horizontal",
    showextrema=False,
)
for body in violin["bodies"]:
    body.set_alpha(0.75)
ax.boxplot(
    observed_water_level,
    positions=[1],
    widths=0.15,
    orientation="horizontal",
    patch_artist=True,
    showfliers=True,
    flierprops={"marker": ".", "markerfacecolor": "none", "markeredgecolor": "black", "markersize": 3, "linestyle": "none", "alpha": 0.6, "zorder": 5},
)

ax.axvline(
    WATER_LEVEL_ALARM_THRESHOLD_CM,
    color="C2",
    linestyle="--",
    label=f"Alarm threshold ({WATER_LEVEL_ALARM_THRESHOLD_CM:.0f} cm)",
)
ax.set_xlabel("Water level [cm]")
ax.set_title(f"Observed water level at {TARGET_STATION_ID}")
ax.set_yticks([])
ax.tick_params(axis="y", left=False, labelleft=False)
ax.grid(alpha=0.25, axis="x")
# ax.text(
#     0.98,
#     0.95,
#     f"{above_alarm_percentage:.1f}% of observations\nabove alarm threshold",
#     transform=ax.transAxes,
#     ha="right",
#     va="top",
#     bbox={"boxstyle": "round,pad=0.3", "facecolor": "white", "edgecolor": "0.5", "alpha": 0.9},
# )
print(f"Number of observations above alarm threshold: {above_alarm_count} ({above_alarm_percentage:.2f}%)")
print(f"Quartiles: Q1={quartiles[0.25]:.2f}, Q2={quartiles[0.50]:.2f}, Q3={quartiles[0.75]:.2f}")
ax.legend()
save_figure(
    fig,
    "target_water_level_distribution",
    caption="Distribution of observed hourly water levels at the target station.",
)

## Table — processed data coverage by station

One row per station in the joined train and test data. `start date` and `end date` are the first and last timestamps with a non-null water-level value; `missing count` is the number of null water level rows; `largest gap (hours)` is the largest elapsed gap between consecutive non-missing water-level observations.

In [ ]:
joined_data = pd.concat(
    [
        pd.read_parquet(raw_train_path, columns=["timestamp", *water_level_columns]),
        pd.read_parquet(raw_test_path, columns=["timestamp", *water_level_columns]),
    ],
    ignore_index=True,
).sort_values("timestamp")
timestamps = pd.to_datetime(joined_data["timestamp"], utc=True)
coverage_rows = []
for column in water_level_columns:
    station_id = column.removesuffix("__water_level")
    water_level = joined_data[column]
    observed_timestamps = timestamps[water_level.notna()]
    largest_gap_hours = (
        observed_timestamps.diff().dt.total_seconds().div(3600).max()
    )
    coverage_rows.append(
        {
            "Station": station_id,
            "Start date": observed_timestamps.min().strftime("%Y-%m-%d %H:%M"),
            "End date": observed_timestamps.max().strftime("%Y-%m-%d %H:%M"),
            # "Row count": len(joined_data),
            "Missing count": int(water_level.isna().sum()),
            "Largest gap (h)": int(largest_gap_hours),
        }
    )

station_coverage = pd.DataFrame(coverage_rows)
display(station_coverage)
save_table(
    station_coverage,
    "station_data_coverage",
    caption=f"Coverage of water level data across stations. (Total row count: {len(joined_data):,})",
    index=False
)

## Using the exports in the thesis

Each `save_*` call prints the float to paste into a chapter, with the path already relative to the thesis repository root — copy it as-is. The figure is authored at exactly `\textwidth`, so `width=\textwidth` scales it by 1.0 and its fonts match the body text.